In [2]:
# necessary imports
import pandas as pd
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import contractions
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import BertTokenizer
import joblib

c:\Users\abdul\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_parquet('../data/processed/customer_support_tickets.parquet')

In [4]:
df.dtypes

Ticket ID                                int64
Customer Name                           object
Customer Email                          object
Customer Age                             int64
Customer Gender                         object
Product Purchased                       object
Date of Purchase                datetime64[ns]
Ticket Type                             object
Ticket Subject                          object
Ticket Description                      object
Ticket Status                           object
Resolution                              object
Ticket Priority                         object
Ticket Channel                          object
First Response Time             datetime64[ns]
Time to Resolution              datetime64[ns]
Customer Satisfaction Rating           float64
dtype: object

In [5]:
# required datetime conversions
df['Date of Purchase'] = pd.to_datetime(df['Date of Purchase'])
df['First Response Time'] = pd.to_datetime(df['First Response Time'])
df['Time to Resolution'] = pd.to_datetime(df['Time to Resolution'])

In [6]:
# Step - 3a
# create a text cleaning function
def clean_text(text):
    text = str(text).lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove contractions
    text = contractions.fix(text)

   # remove placeholders
    text = re.sub(r'\{.*?\}', ' ', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['clean_description'] = (
    df['Ticket Description']
    .apply(clean_text)
)

df['clean_ticket_subject'] = (
    df['Ticket Subject']
    .apply(clean_text)
)

In [7]:
# df.head()

In [8]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\abdul\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abdul\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\abdul\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\abdul\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [9]:
# Step - 3b
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Create a function to create tokens
def preprocess_text(text):
    text = text.lower()

    words = text.split()   # simple tokenization

    words = [
        word for word in words
        if word not in stop_words
    ]

    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)

# create tokens
df['processed_description'] = (
    df['clean_description']
    .apply(preprocess_text)
)

df['processed_ticket_subject'] = (
    df['clean_ticket_subject']
    .apply(preprocess_text)
)

In [10]:
df[
[
    'Ticket Description',
    'processed_description',
    'Ticket Status',
    'processed_ticket_subject'
]
].head()

,Ticket Description,processed_description,Ticket Status,processed_ticket_subject
0,I'm having an issue with the {product_purchase...,issue please assist billing zip code 71701 app...,Pending Customer Response,product setup
1,I'm having an issue with the {product_purchase...,issue please assist need change existing produ...,Pending Customer Response,peripheral compatibility
2,I'm facing a problem with my {product_purchase...,facing problem turning working fine yesterday ...,Closed,network problem
3,I'm having an issue with the {product_purchase...,issue please assist problem interested would l...,Closed,account access
4,I'm having an issue with the {product_purchase...,issue please assist note seller responsible da...,Closed,data loss


In [11]:
# Most common words occuring in ticket description (Top-20)
from collections import Counter

all_words = " ".join(df['processed_description']).split()

Counter(all_words).most_common(20)

[('issue', 11828),
 ('please', 8811),
 ('assist', 6251),
 ('problem', 2609),
 ('product', 2476),
 ('update', 1865),
 ('data', 1649),
 ('device', 1566),
 ('software', 1530),
 ('account', 1475),
 ('step', 1455),
 ('time', 1216),
 ('would', 1212),
 ('help', 1181),
 ('persists', 1178),
 ('noticed', 1178),
 ('work', 1170),
 ('resolve', 1166),
 ('unable', 1075),
 ('could', 1066)]

## Text Preprocessing Summary

The ticket descriptions were preprocessed to improve text quality and reduce noise. The preprocessing pipeline included lowercasing, removal of punctuation, HTML artifacts, URLs, and placeholder tokens. Stopwords were removed and lemmatization was applied to normalize words to their root forms. The resulting processed text was used for downstream feature extraction using TF-IDF and transformer-based embeddings.


In [13]:
# Compute TF-IDF vectors

df['combined_text'] = (
    df['processed_ticket_subject'] + ' ' +
    df['processed_description']
)

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_tfidf = tfidf.fit_transform(
    df['combined_text']
)

In [14]:
print(X_tfidf.shape)

(8469, 5000)


In [15]:
# Bert Tokenization
tokenizer = BertTokenizer.from_pretrained(
    'bert-base-uncased'
)

In [16]:
sample = df['combined_text'][0]

encoded = tokenizer(
    sample,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

print(encoded.keys())

KeysView({'input_ids': tensor([[  101,  4031, 16437,  3277,  3531,  6509, 25640, 14101,  3642,  6390,
         19841,  2487,  9120,  7303,  4037,  4769,  3531,  3313,  4638, 10373,
          4769,  2699, 13460, 23416,  2075,  3357,  3855,  5310,  6410,  3277,
         29486,  2015,   102,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,

In [17]:
print(encoded['input_ids'].shape)
print(encoded['attention_mask'].shape)

torch.Size([1, 128])
torch.Size([1, 128])


In [18]:
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,...,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating,clean_description,clean_ticket_subject,processed_description,processed_ticket_subject,combined_text
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,...,Critical,Social media,2023-06-01 12:15:36,NaT,NaN,i am having an issue with the please assist yo...,product setup,issue please assist billing zip code 71701 app...,product setup,product setup issue please assist billing zip ...
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,...,Critical,Chat,2023-06-01 16:45:38,NaT,NaN,i am having an issue with the please assist if...,peripheral compatibility,issue please assist need change existing produ...,peripheral compatibility,peripheral compatibility issue please assist n...
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,...,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0,i am facing a problem with my the is not turni...,network problem,facing problem turning working fine yesterday ...,network problem,network problem facing problem turning working...
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,...,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0,i am having an issue with the please assist if...,account access,issue please assist problem interested would l...,account access,account access issue please assist problem int...
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,...,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0,i am having an issue with the please assist no...,data loss,issue please assist note seller responsible da...,data loss,data loss issue please assist note seller resp...


In [19]:
# save processed text description

df.to_parquet(
    "../data/processed/text_processed.parquet",
    index=False
)

In [20]:
# save tfidf vectors
joblib.dump(
    tfidf,
    "../models/tfidf_vectorizer.pkl"
)

['../models/tfidf_vectorizer.pkl']

In [21]:
# save bert tokenizer
tokenizer.save_pretrained(
    "../models/bert_tokenizer"
)

('../models/bert_tokenizer\\tokenizer_config.json',
 '../models/bert_tokenizer\\tokenizer.json')